In [1]:
import pandas as pd
import numpy as np

## Yield Prediction using Gradient Boosting and PyTorch

### 1. Data Loading and Initial Exploration

First, we'll load the dataset and take a look at its structure, data types, and a few sample rows.

In [2]:
try:
    df = pd.read_csv('/content/production_unified_imputed.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: The file '/content/production_unified_imputed.csv' was not found. Please ensure it's uploaded correctly.")

    df = None


if df is not None:
    print("\nFirst 5 rows of the dataset:")
    display(df.head())

    print("\nDataFrame Info:")
    df.info()

Dataset loaded successfully.

First 5 rows of the dataset:


,crop,year,state,district,season,area,production,yield,data_source,annual_rainfall,fertilizer,pesticide,crop_type,area_unit,production_unit,yield_unit
0,wheat,2014,uttar pradesh,ghaziabad,Rabi,26619.0,86645.0,3.255,area_production,1247.199725,1.248187e+06,2438.165184,cereals,Hectare,Tonnes,Tonnes/Hectare
1,urad,2014,uttar pradesh,ghaziabad,Summer,22.0,12.0,0.545,area_production,1247.339375,1.235179e+06,2420.293698,pulses,Hectare,Tonnes,Tonnes/Hectare
2,urad,2014,uttar pradesh,ghaziabad,Kharif,19.0,10.0,0.526,area_production,1247.339396,1.235177e+06,2420.290923,pulses,Hectare,Tonnes,Tonnes/Hectare
3,sugarcane,2014,uttar pradesh,ghaziabad,Kharif,11136.0,750076.0,67.356,area_production,1247.223783,1.245441e+06,2437.372883,sugar,Hectare,Tonnes,Tonnes/Hectare
4,rice,2014,uttar pradesh,ghaziabad,Kharif,8650.0,23727.0,2.743,area_production,1247.278389,1.240637e+06,2428.557661,cereals,Hectare,Tonnes,Tonnes/Hectare



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440962 entries, 0 to 440961
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   crop             440962 non-null  object 
 1   year             440962 non-null  int64  
 2   state            440962 non-null  object 
 3   district         440962 non-null  object 
 4   season           440962 non-null  object 
 5   area             440962 non-null  float64
 6   production       440962 non-null  float64
 7   yield            440962 non-null  float64
 8   data_source      440962 non-null  object 
 9   annual_rainfall  440962 non-null  float64
 10  fertilizer       440962 non-null  float64
 11  pesticide        440962 non-null  float64
 12  crop_type        440962 non-null  object 
 13  area_unit        440962 non-null  object 
 14  production_unit  440962 non-null  object 
 15  yield_unit       440962 non-null  object 
dtypes: float64(6), int64(

### 2. Data Preprocessing

Now, let's prepare the data for modeling. This typically involves:
- Identifying categorical and numerical features.
- Encoding categorical features using techniques like one-hot encoding.
- Splitting the data into training and testing sets.

In [3]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'yield' in numerical_features:
    numerical_features.remove('yield')

print(f"Categorical features: {categorical_features}")
print(f"Numerical features (excluding target 'yield'): {numerical_features}")

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("\nDataFrame after one-hot encoding:")
display(df_encoded.head())
print(f"New shape of DataFrame: {df_encoded.shape}")

Categorical features: ['crop', 'state', 'district', 'season', 'data_source', 'crop_type', 'area_unit', 'production_unit', 'yield_unit']
Numerical features (excluding target 'yield'): ['year', 'area', 'production', 'annual_rainfall', 'fertilizer', 'pesticide']

DataFrame after one-hot encoding:


,year,area,production,yield,annual_rainfall,fertilizer,pesticide,crop_arcanut (processed),crop_arecanut,crop_arhar/tur,...,data_source_des_district,crop_type_drugs and narcotics,crop_type_fiber crops,crop_type_fruits,crop_type_oilseeds,crop_type_plantation crops,crop_type_pulses,crop_type_spices,crop_type_sugar,crop_type_vegetable
0,2014,26619.0,86645.0,3.255,1247.199725,1.248187e+06,2438.165184,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2014,22.0,12.0,0.545,1247.339375,1.235179e+06,2420.293698,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,2014,19.0,10.0,0.526,1247.339396,1.235177e+06,2420.290923,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,2014,11136.0,750076.0,67.356,1247.223783,1.245441e+06,2437.372883,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,2014,8650.0,23727.0,2.743,1247.278389,1.240637e+06,2428.557661,False,False,False,...,False,False,False,False,False,False,False,False,False,False


New shape of DataFrame: (440962, 926)


### 3. Splitting Data into Training and Testing Sets

Now, we'll define our features (X) and target variable (y), and then split them into training and testing sets to evaluate our models effectively.

In [4]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('yield', axis=1)
y = df_encoded['yield']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (352769, 925)
Shape of X_test: (88193, 925)
Shape of y_train: (352769,)
Shape of y_test: (88193,)


### 4. Preparing Data for PyTorch

To use PyTorch, we need to convert our numpy arrays (from pandas DataFrames/Series) into PyTorch tensors. We'll also create `DataLoader` objects for efficient batch processing during training.

In [5]:
# import torch
# from torch.utils.data import TensorDataset, DataLoader


# X_train_np = X_train.values.astype(np.float32)
# X_test_np = X_test.values.astype(np.float32)
# y_train_np = y_train.values.astype(np.float32)
# y_test_np = y_test.values.astype(np.float32)

# X_train_tensor = torch.tensor(X_train_np)
# X_test_tensor = torch.tensor(X_test_np)
# y_train_tensor = torch.tensor(y_train_np).unsqueeze(1)
# y_test_tensor = torch.tensor(y_test_np).unsqueeze(1)

# print(f"X_train_tensor shape: {X_train_tensor.shape}")
# print(f"y_train_tensor shape: {y_train_tensor.shape}")

# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# batch_size = 64
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# print(f"\nNumber of training batches: {len(train_loader)}")
# print(f"Number of testing batches: {len(test_loader)}")

### 5. Define PyTorch Model Architecture

Now we will define a simple Multi-Layer Perceptron (MLP) using PyTorch's `nn.Module` for our yield prediction task. Since this is a regression problem, the output layer will have a single neuron without an activation function, suitable for predicting continuous values.

In [6]:
# import torch.nn as nn
# import torch.optim as optim

# class YieldPredictor(nn.Module):
#     def __init__(self, input_size):
#         super(YieldPredictor, self).__init__()
#         self.fc1 = nn.Linear(input_size, 256)
#         self.relu1 = nn.ReLU()
#         self.dropout1 = nn.Dropout(0.3)
#         self.fc2 = nn.Linear(256, 128)
#         self.relu2 = nn.ReLU()
#         self.dropout2 = nn.Dropout(0.3)
#         self.fc3 = nn.Linear(128, 64)
#         self.relu3 = nn.ReLU()
#         self.dropout3 = nn.Dropout(0.2)
#         self.fc4 = nn.Linear(64, 1)

#     def forward(self, x):
#         x = self.fc1(x)
#         x = self.relu1(x)
#         x = self.dropout1(x)
#         x = self.fc2(x)
#         x = self.relu2(x)
#         x = self.dropout2(x)
#         x = self.fc3(x)
#         x = self.relu3(x)
#         x = self.dropout3(x)
#         x = self.fc4(x)
#         return x

# input_size = X_train_tensor.shape[1]
# model = YieldPredictor(input_size)

# criterion = nn.MSELoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# print(model)
# print(f"\nLoss function: {criterion}")
# print(f"Optimizer: {optimizer}")

### 6. Train the PyTorch Model

We will now train the defined `YieldPredictor` model using our `train_loader`. We'll monitor the training loss and evaluate the model's performance on the test set after training.

In [7]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Using device: {device}")

# num_epochs = 10

# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     for inputs, labels in train_loader:
#         inputs, labels = inputs.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item() * inputs.size(0)

#     epoch_loss = running_loss / len(train_dataset)
#     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

# print("\nTraining finished!")

### 7. Evaluate the PyTorch Model

After training, we will evaluate the model's performance on the unseen test dataset using metrics like Mean Squared Error (MSE) and Root Mean Squared Error (RMSE).

In [8]:
# model.eval()
# predictions = []
# true_labels = []

# with torch.no_grad():
#     for inputs, labels in test_loader:
#         inputs, labels = inputs.to(device), labels.to(device)
#         outputs = model(inputs)
#         predictions.extend(outputs.cpu().numpy())
#         true_labels.extend(labels.cpu().numpy())

# predictions = np.array(predictions).flatten()
# true_labels = np.array(true_labels).flatten()

# mse = mean_squared_error(true_labels, predictions)
# rmse = np.sqrt(mse)
# r2 = r2_score(true_labels, predictions)

# print(f"\nModel Evaluation on Test Set:")
# print(f"Mean Squared Error (MSE): {mse:.4f}")
# print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
# print(f"R-squared (R2): {r2:.4f}")

### 8. Yield Prediction with Gradient Boosting (XGBoost)

Since the PyTorch model's performance was low, let's now implement a gradient boosting model, specifically XGBoost, which is known for its strong performance on tabular data. We will train it on the same training data and evaluate its performance.

In [9]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',      # much faster than default 'exact'
    device='cuda'            # only if you've enabled a GPU runtime
)

print("Training XGBoost Regressor...")

xgb_model.fit(X_train, y_train)

print("XGBoost training complete. Evaluating on test set...")

xgb_predictions = xgb_model.predict(X_test)

mse_xgb = mean_squared_error(y_test, xgb_predictions)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_test, xgb_predictions)

print(f"\nXGBoost Model Evaluation on Test Set:")
print(f"Mean Squared Error (MSE): {mse_xgb:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
print(f"R-squared (R2): {r2_xgb:.4f}")

Training XGBoost Regressor...
XGBoost training complete. Evaluating on test set...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [20:53:58] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



XGBoost Model Evaluation on Test Set:
Mean Squared Error (MSE): 44.4628
Root Mean Squared Error (RMSE): 6.6680
R-squared (R2): 0.9100


### 9. Summary and Comparison of Models

We've trained and evaluated both a PyTorch neural network and an XGBoost regressor for yield prediction. Let's compare their performance metrics.

In [10]:
print("\n--- Model Performance Summary ---")

print(f"\nXGBoost Model:")
print(f"  Mean Squared Error (MSE): {mse_xgb:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
print(f"  R-squared (R2): {r2_xgb:.4f}")

# Placeholder for LightGBM and CatBoost summaries
# These will be updated after their models are trained.

# Initialize R2 scores for comparison (currently only XGBoost)
models_r2 = {
    'XGBoost': r2_xgb
}

# Determine the best model based on R-squared (initial comparison)
best_model_name = max(models_r2, key=models_r2.get)
best_r2_score = models_r2[best_model_name]

print(f"\nConclusion: The {best_model_name} model performed best among the current models with an R-squared of {best_r2_score:.4f}.")

print("\nThis concludes the yield prediction analysis using gradient boosting. If you have further questions or want to explore other aspects, please let me know!")


--- Model Performance Summary ---

XGBoost Model:
  Mean Squared Error (MSE): 44.4628
  Root Mean Squared Error (RMSE): 6.6680
  R-squared (R2): 0.9100

Conclusion: The XGBoost model performed best among the current models with an R-squared of 0.9100.

This concludes the yield prediction analysis using gradient boosting. If you have further questions or want to explore other aspects, please let me know!


### 10. Exploring Other Gradient Boosting Methods and Hyperparameter Tuning

While XGBoost performed well, other gradient boosting libraries like LightGBM and CatBoost are also popular and can offer different performance characteristics, especially in terms of speed and handling of categorical features. Furthermore, hyperparameter tuning is crucial for extracting the best possible performance from any machine learning model.

Let's first briefly mention LightGBM and CatBoost, and then proceed with hyperparameter tuning for our XGBoost model.

#### B. LightGBM Model

LightGBM is another popular gradient boosting framework that uses tree-based learning algorithms. It's known for its speed and efficiency, especially with large datasets, and also supports GPU acceleration.

In [11]:
import lightgbm as lgb
import torch
from torch.utils.data import TensorDataset, DataLoader

print("Training LightGBM Regressor...")

lgbm_model = lgb.LGBMRegressor(
    objective='regression_l1', # MAE objective, often more robust to outliers
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1, # No limit on tree depth
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    device='gpu' if torch.cuda.is_available() else 'cpu' # Use GPU if available
)

lgbm_model.fit(X_train, y_train)

print("LightGBM training complete. Evaluating on test set...")

lgbm_predictions = lgbm_model.predict(X_test)

mse_lgbm = mean_squared_error(y_test, lgbm_predictions)
rmse_lgbm = np.sqrt(mse_lgbm)
r2_lgbm = r2_score(y_test, lgbm_predictions)

print(f"\nLightGBM Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_lgbm:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_lgbm:.4f}")
print(f"  R-squared (R2): {r2_lgbm:.4f}")

Training LightGBM Regressor...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3069
[LightGBM] [Info] Number of data points in the train set: 352769, number of used features: 889
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 11 dense feature groups (4.04 MB) transferred to GPU in 0.007598 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 1.056000
LightGBM training complete. Evaluating on test set...

LightGBM Model Evaluation on Test Set:
  Mean Squared Error (MSE): 321.1490
  Root Mean Squared Error (RMSE): 17.9206
  R-squared (R2): 0.3503


#### A. Hyperparameter Tuning for LightGBM

In [12]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print("Starting Hyperparameter Tuning for LightGBM using RandomizedSearchCV...")

lgbm_param_dist = {
    'n_estimators': stats.randint(100, 1000),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'num_leaves': stats.randint(20, 60),
    'max_depth': stats.randint(5, 15),
    'min_child_samples': stats.randint(10, 50),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'reg_alpha': stats.loguniform(1e-5, 1),
    'reg_lambda': stats.loguniform(1e-5, 1)
}

lgbm_random_search = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(objective='regression_l1', random_state=42, n_jobs=-1, device='gpu' if torch.cuda.is_available() else 'cpu'),
    param_distributions=lgbm_param_dist,
    n_iter=20, # Reduced iterations for demonstration, can be increased
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

lgbm_random_search.fit(X_train, y_train)

print("\nLightGBM Hyperparameter tuning complete.")
print(f"Best parameters found for LightGBM: {lgbm_random_search.best_params_}")
print(f"Best cross-validation MSE for LightGBM: {-lgbm_random_search.best_score_:.4f}")

best_lgbm_model = lgbm_random_search.best_estimator_

best_lgbm_predictions = best_lgbm_model.predict(X_test)

mse_best_lgbm = mean_squared_error(y_test, best_lgbm_predictions)
rmse_best_lgbm = np.sqrt(mse_best_lgbm)
r2_best_lgbm = r2_score(y_test, best_lgbm_predictions)

print(f"\nBest Tuned LightGBM Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_best_lgbm:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_best_lgbm:.4f}")
print(f"  R-squared (R2): {r2_best_lgbm:.4f}")

Starting Hyperparameter Tuning for LightGBM using RandomizedSearchCV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3085
[LightGBM] [Info] Number of data points in the train set: 352769, number of used features: 897
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 11 dense feature groups (4.04 MB) transferred to GPU in 0.008066 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 1.056000

LightGBM Hyperparameter tuning complete.
Best parameters found for LightGBM: {'colsample_bytree': np.float64(0.8604308102007778), 'learning_rate': np.float64(0.2246499798407724), 'max_depth': 13, 'min_child_samples': 17

#### C. CatBoost Model

CatBoost is another powerful open-source gradient boosting library that excels at handling categorical features automatically. It also offers GPU support for faster training.

In [13]:
!pip install catboost

In [14]:
from catboost import CatBoostRegressor

print("Training CatBoost Regressor...")

cat_features_indices = [X.columns.get_loc(col) for col in categorical_features if col in X.columns]

catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=0, # Suppress verbose output
    early_stopping_rounds=50,
    task_type='GPU' if torch.cuda.is_available() else 'CPU' # Use GPU if available
)

catboost_model.fit(X_train, y_train,
                    # cat_features=cat_features_indices, # CatBoost can handle categorical features directly
                    # Although our data is one-hot encoded, if original categorical features were passed,
                    # this would be beneficial. For OHE, it's not strictly necessary.
                    # For this problem, we are using OHE features only.
                    early_stopping_rounds=50 # Use early stopping for faster training
                   )

print("CatBoost training complete. Evaluating on test set...")

catboost_predictions = catboost_model.predict(X_test)

mse_catboost = mean_squared_error(y_test, catboost_predictions)
rmse_catboost = np.sqrt(mse_catboost)
r2_catboost = r2_score(y_test, catboost_predictions)

print(f"\nCatBoost Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_catboost:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_catboost:.4f}")
print(f"  R-squared (R2): {r2_catboost:.4f}")

Training CatBoost Regressor...


Default metric period is 5 because R2 is/are not implemented for GPU
Metric R2 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


CatBoost training complete. Evaluating on test set...

CatBoost Model Evaluation on Test Set:
  Mean Squared Error (MSE): 47.6545
  Root Mean Squared Error (RMSE): 6.9032
  R-squared (R2): 0.9036


#### B. Hyperparameter Tuning for CatBoost

In [15]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print("Starting Hyperparameter Tuning for CatBoost using RandomizedSearchCV...")

catboost_param_dist = {
    'iterations': stats.randint(100, 1000),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'depth': stats.randint(4, 10),
    'l2_leaf_reg': stats.loguniform(1e-3, 10),
    'subsample': stats.uniform(0.6, 0.4)   # correct param for MVS bootstrap
}
catboost_random_search = RandomizedSearchCV(
    estimator=CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        verbose=0,
        early_stopping_rounds=50,
        bootstrap_type='MVS',
        task_type='GPU' if torch.cuda.is_available() else 'CPU'
    ),
    param_distributions=catboost_param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=1 if torch.cuda.is_available() else -1   # avoid GPU contention
)
catboost_random_search.fit(X_train, y_train)

print("\nCatBoost Hyperparameter tuning complete.")
print(f"Best parameters found for CatBoost: {catboost_random_search.best_params_}")
print(f"Best cross-validation MSE for CatBoost: {-catboost_random_search.best_score_:.4f}")

best_catboost_model = catboost_random_search.best_estimator_

best_catboost_predictions = best_catboost_model.predict(X_test)

mse_best_catboost = mean_squared_error(y_test, best_catboost_predictions)
rmse_best_catboost = np.sqrt(mse_best_catboost)
r2_best_catboost = r2_score(y_test, best_catboost_predictions)

print(f"\nBest Tuned CatBoost Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_best_catboost:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_best_catboost:.4f}")
print(f"  R-squared (R2): {r2_best_catboost:.4f}")

Starting Hyperparameter Tuning for CatBoost using RandomizedSearchCV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

CatBoost Hyperparameter tuning complete.
Best parameters found for CatBoost: {'depth': 8, 'iterations': 437, 'l2_leaf_reg': np.float64(0.1441346937111033), 'learning_rate': np.float64(0.07357031542417702), 'subsample': np.float64(0.9861021229056552)}
Best cross-validation MSE for CatBoost: 243.7054

Best Tuned CatBoost Model Evaluation on Test Set:
  Mean Squared Error (MSE): 49.7904
  Root Mean Squared Error (RMSE): 7.0562
  R-squared (R2): 0.8993


#### A. Hyperparameter Tuning for XGBoost

Hyperparameter tuning involves finding the best combination of parameters for a model that minimizes a chosen error metric (e.g., MSE). We'll use `RandomizedSearchCV` from `sklearn` to efficiently search through a defined range of hyperparameters. This method randomly samples combinations from the parameter space, which is generally more efficient than a full grid search for large search spaces.

In [16]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print("Starting Hyperparameter Tuning for XGBoost using RandomizedSearchCV...")

param_dist = {
    'n_estimators': stats.randint(100, 1000),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'max_depth': stats.randint(3, 10),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'gamma': stats.uniform(0, 0.5),
    'reg_alpha': stats.loguniform(1e-5, 1),
    'reg_lambda': stats.loguniform(1e-5, 1)
}

random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=1,
        tree_method='hist',
        device='cuda',
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1                   # <-- changed: avoid GPU contention, let XGBoost's own GPU execution do the work
)

random_search.fit(X_train, y_train)

print("\nHyperparameter tuning complete.")
print(f"Best parameters found: {random_search.best_params_}")
print(f"Best cross-validation MSE: {-random_search.best_score_:.4f}")

best_xgb_model = random_search.best_estimator_
best_xgb_predictions = best_xgb_model.predict(X_test)

mse_best_xgb = mean_squared_error(y_test, best_xgb_predictions)
rmse_best_xgb = np.sqrt(mse_best_xgb)
r2_best_xgb = r2_score(y_test, best_xgb_predictions)

print(f"\nBest Tuned XGBoost Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_best_xgb:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_best_xgb:.4f}")
print(f"  R-squared (R2): {r2_best_xgb:.4f}")

Starting Hyperparameter Tuning for XGBoost using RandomizedSearchCV...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.749816047538945, gamma=0.4753571532049581, learning_rate=0.1205712628744377, max_depth=7, n_estimators=120, reg_alpha=6.0268891286825045e-05, reg_lambda=6.025215736203858e-05, subsample=0.6232334448672797; total time=  24.6s
[CV] END colsample_bytree=0.749816047538945, gamma=0.4753571532049581, learning_rate=0.1205712628744377, max_depth=7, n_estimators=120, reg_alpha=6.0268891286825045e-05, reg_lambda=6.025215736203858e-05, subsample=0.6232334448672797; total time=  22.5s
[CV] END colsample_bytree=0.749816047538945, gamma=0.4753571532049581, learning_rate=0.1205712628744377, max_depth=7, n_estimators=120, reg_alpha=6.0268891286825045e-05, reg_lambda=6.025215736203858e-05, subsample=0.6232334448672797; total time=  21.7s
[CV] END colsample_bytree=0.9464704583099741, gamma=0.3005575058716044, learning_rate=0.11114989443094977, ma

In [17]:
print("\n--- Model Performance Summary (Updated) ---")

models_r2 = {}

# XGBoost Model (Initial)
if 'mse_xgb' in locals():
    print(f"\nXGBoost Model (Initial):")
    print(f"  Mean Squared Error (MSE): {mse_xgb:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_xgb:.4f}")
    print(f"  R-squared (R2): {r2_xgb:.4f}")
    models_r2['XGBoost'] = r2_xgb

# XGBoost Model (Tuned)
if 'mse_best_xgb' in locals():
    print(f"\nXGBoost Model (Tuned):")
    print(f"  Mean Squared Error (MSE): {mse_best_xgb:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_best_xgb:.4f}")
    print(f"  R-squared (R2): {r2_best_xgb:.4f}")
    models_r2['XGBoost (Tuned)'] = r2_best_xgb

# LightGBM Model (Initial)
if 'mse_lgbm' in locals():
    print(f"\nLightGBM Model (Initial):")
    print(f"  Mean Squared Error (MSE): {mse_lgbm:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_lgbm:.4f}")
    print(f"  R-squared (R2): {r2_lgbm:.4f}")
    models_r2['LightGBM'] = r2_lgbm

# LightGBM Model (Tuned)
if 'mse_best_lgbm' in locals():
    print(f"\nLightGBM Model (Tuned):")
    print(f"  Mean Squared Error (MSE): {mse_best_lgbm:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_best_lgbm:.4f}")
    print(f"  R-squared (R2): {r2_best_lgbm:.4f}")
    models_r2['LightGBM (Tuned)'] = r2_best_lgbm

# CatBoost Model (Initial)
if 'mse_catboost' in locals():
    print(f"\nCatBoost Model (Initial):")
    print(f"  Mean Squared Error (MSE): {mse_catboost:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_catboost:.4f}")
    print(f"  R-squared (R2): {r2_catboost:.4f}")
    models_r2['CatBoost'] = r2_catboost

# CatBoost Model (Tuned)
if 'mse_best_catboost' in locals():
    print(f"\nCatBoost Model (Tuned):")
    print(f"  Mean Squared Error (MSE): {mse_best_catboost:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_best_catboost:.4f}")
    print(f"  R-squared (R2): {r2_best_catboost:.4f}")
    models_r2['CatBoost (Tuned)'] = r2_best_catboost

# Determine the best model based on R-squared
if models_r2:
    best_model_name = max(models_r2, key=models_r2.get)
    best_r2_score = models_r2[best_model_name]
    print(f"\nConclusion: The {best_model_name} model performed best with an R-squared of {best_r2_score:.4f}.")
else:
    print("\nConclusion: No model results are available for comparison yet. Please run the model training and tuning cells.")

print("\nThis concludes the yield prediction analysis using various gradient boosting techniques. If you have further questions or want to explore other aspects, please let me know!")


--- Model Performance Summary (Updated) ---

XGBoost Model (Initial):
  Mean Squared Error (MSE): 44.4628
  Root Mean Squared Error (RMSE): 6.6680
  R-squared (R2): 0.9100

XGBoost Model (Tuned):
  Mean Squared Error (MSE): 39.0068
  Root Mean Squared Error (RMSE): 6.2455
  R-squared (R2): 0.9211

LightGBM Model (Initial):
  Mean Squared Error (MSE): 321.1490
  Root Mean Squared Error (RMSE): 17.9206
  R-squared (R2): 0.3503

LightGBM Model (Tuned):
  Mean Squared Error (MSE): 205.3756
  Root Mean Squared Error (RMSE): 14.3309
  R-squared (R2): 0.5845

CatBoost Model (Initial):
  Mean Squared Error (MSE): 47.6545
  Root Mean Squared Error (RMSE): 6.9032
  R-squared (R2): 0.9036

CatBoost Model (Tuned):
  Mean Squared Error (MSE): 49.7904
  Root Mean Squared Error (RMSE): 7.0562
  R-squared (R2): 0.8993

Conclusion: The XGBoost (Tuned) model performed best with an R-squared of 0.9211.

This concludes the yield prediction analysis using various gradient boosting techniques. If you have 